In [1]:
import tensorflow as tf
from keras.utils import to_categorical
from keras import layers, models, optimizers, metrics


import sys
sys.path.append('..')
from scripts.prepare_data import download_data, preproces_baseline_forest
from sklearn.preprocessing import StandardScaler

2026-01-10 16:17:38.987661: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-10 16:17:39.020721: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-10 16:17:39.804783: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/kuba/RNN-ECG-analysis/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning

In [2]:
df_base_preproc = download_data()
X_train, X_test, y_train, y_test = preproces_baseline_forest(df_base_preproc)

In [9]:
time_steps = X_train.shape[1]
features = 1

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_preproc = X_train.reshape((X_train.shape[0], time_steps, features))
X_test_preproc = X_test.reshape((X_test.shape[0], time_steps, features))

batch_size = 32

y_train_preproc = to_categorical(y_train)
y_test_preproc = to_categorical(y_test)

train_ds = tf.data.Dataset.from_tensor_slices((X_train_preproc, y_train_preproc)) \
    .shuffle(1000) \
    .batch(batch_size) \
    .prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_test_preproc, y_test_preproc)) \
    .batch(batch_size) \
    .prefetch(tf.data.AUTOTUNE)

In [ ]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.LSTM(512),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro"), metrics.F1Score(average="micro")])

history = deep_rnn.fit(train_ds, validation_data=val_ds, epochs=10)